# Hitmakers — Full Model Pipeline & Comparison

Models compared with **5-fold stratified cross-validation** on `df_artists_final.csv`
(759 artists × 26 features).


 **Target** `top_20_hitmaker` (binary: 1 = hitmaker, 0 = one-hit wonder) 

 **Split**  80 / 20 stratified (`random_state=42`) 
 
 **Class balance**  ~57 % one-hit wonders · ~43 % hitmakers 

### Models
| Model | Type | Key properties |
|---|---|---|
| Stratified Baseline | Baseline | Predicts class ratio (~43 % hitmaker) |
| AdaBoost Linear | Adaptive Boosting | `SGDClassifier(loss='log_loss')` weak learner, StandardScaler required |
| AdaBoost Tree | Adaptive Boosting | Decision-tree stumps (Freund & Schapire 1997) |
| Random Forest | Ensemble (bagging) | `class_weight='balanced'` for imbalance |
| LightGBM | Gradient Boosting | Histogram-based splits, depth-limited |
| XGBoost | Gradient Boosting | L1/L2 regularization, column subsampling |
| CatBoost | Gradient Boosting | Ordered boosting, symmetric trees, built-in regularization |

### Per-model pipeline:
| Step | Name | Purpose |
|:----:|------|---------|
| 1 | Full-feature Optuna | Tune hyperparams on all 26 features; penalized score: AUC − λ × gap → `params_full` |
| 2 | CV feature importance | CV SHAP for tree models · CV permutation importance for AdaBoost Linear |
| 3 | Genre consolidation | Tree models: keep genres above mean SHAP; perm models: keep importance > 0; remainder → `artist_genre_other` |
| 4 | Forward selection | Greedy addition on consolidated set ordered by Step 2 importance; centrality features included in pool |
| 5 | Re-tune + winner | Two candidates: `n_peak` (max CV AUC) and `n_gap` (min overfit gap); Optuna re-tune on both; winner by penalized score; MIN_N = 5 floor |
| 6 | Centrality ablation | Post-hoc: drop any centrality subset that improves raw CV AUC on the winning feature set |
| 7 | Final evaluation | Fit on full training set; ROC, confusion matrix, calibration curve, PR curve, lift curve |
| 8 | OOF threshold tuning | Leakage-safe decision threshold derived from out-of-fold training predictions |


Cross-model section: metrics table, CV AUC ± std bar chart, lift overlay, disagreement table, SHAP feature importance heatmaps.

In [2]:
import shap
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
LAM             = 0.3
N_SPLITS        = 5
N_TRIALS_FULL   = 30
N_TRIALS_RETUNE = 30
RANDOM_STATE    = 42

MODEL_NAMES = [
    'AdaBoost Linear',
    'AdaBoost Tree',
    'Random Forest',
    'LightGBM',
    'XGBoost',
    'CatBoost',
]

# SGDClassifier (AdaBoost Linear base) is scale-sensitive.
# Tree-based models are scale-invariant.
MODEL_NEEDS_SCALER = {name: (name == 'AdaBoost Linear') for name in MODEL_NAMES}

In [ ]:
df = pd.read_csv('df_artists_final.csv', index_col=0).reset_index()
X  = df.drop(columns=['top_20_hitmaker'])
y  = df['top_20_hitmaker']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f'Full dataset:  {df.shape}')
print(f'Train / Test:  {X_train.shape[0]} / {X_test.shape[0]}')
print(f'Features:      {X.shape[1]}')
print(f'Class balance (full):')
print(y.value_counts(normalize=True).round(3).rename({0.0: 'One-hit Wonder', 1.0: 'Hitmaker'}))

## EDA

**Hitmaker rate by genre:** Red bars = genres above the overall hitmaker rate. Motivates genre consolidation in Step 3 — high-signal genres kept separately, low-signal genres merged into `artist_genre_other`.

**Centrality distributions** (training set only): histograms by class + Pearson correlations with target provide a quick linear signal check for network-position features.

In [ ]:
genre_cols   = [c for c in X.columns if c.startswith('artist_genre_')]
overall_mean = y.mean()
rates = []
for col in genre_cols:
    mask = df[col] == 1
    if mask.sum() == 0:
        continue
    rates.append({
        'Genre': col.replace('artist_genre_', ''),
        'Hitmaker Rate': df.loc[mask, 'top_20_hitmaker'].mean(),
        'N': mask.sum(),
    })

df_rates = pd.DataFrame(rates).sort_values('Hitmaker Rate', ascending=True)
fig, ax  = plt.subplots(figsize=(9, 7))
colors   = ['#d62728' if r > overall_mean else '#aec7e8' for r in df_rates['Hitmaker Rate']]
bars     = ax.barh(df_rates['Genre'], df_rates['Hitmaker Rate'], color=colors)
ax.axvline(overall_mean, color='black', linestyle='--', lw=1.5,
           label=f'Overall mean ({overall_mean:.2f})')
for bar, n in zip(bars, df_rates['N']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'n={n}', va='center', fontsize=8)
ax.set_xlabel('Hitmaker Rate')
ax.set_title('Hitmaker Rate by Genre')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
cent_cols = [c for c in X.columns if 'centrality' in c]
df_eda    = X_train.copy()
df_eda['top_20_hitmaker'] = y_train.values

fig, axes = plt.subplots(1, len(cent_cols), figsize=(5 * len(cent_cols), 5))
for ax, col in zip(axes, cent_cols):
    for label, grp in df_eda.groupby('top_20_hitmaker'):
        lbl = 'Hitmaker' if label == 1 else 'One-hit Wonder'
        ax.hist(grp[col].dropna(), bins=25, alpha=0.6, label=lbl, density=True)
    short = col.replace('_centrality_top20_rolling5', '').replace('_', ' ')
    ax.set_title(short, fontsize=10)
    ax.set_xlabel('Value'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Centrality Distribution by Class (Train Set)', fontsize=13)
plt.tight_layout(); plt.show()

print('Pearson correlation with top_20_hitmaker (train set):')
print(df_eda[cent_cols + ['top_20_hitmaker']].corr()['top_20_hitmaker']
      .drop('top_20_hitmaker').round(3))

## Model setup

`build_model(name, params)` constructs any of the 6 classifiers with a uniform sklearn interface (`fit` / `predict_proba`). AdaBoost Linear uses `SGDClassifier(loss='log_loss')` (scale-sensitive); RF uses `class_weight='balanced'` for the 57/43 imbalance.

`make_optuna_objective` returns a penalized objective: **CV AUC − λ × overfit gap**. `cv_evaluate` returns mean ± std across 5 folds. Both include try/except for the rare AdaBoost "base estimator worse than random" error.

In [ ]:
def build_model(name, params):
    if name == 'AdaBoost Linear':
        base = SGDClassifier(
            loss='log_loss',
            alpha=params.get('alpha', 1e-4),
            penalty=params.get('penalty', 'l2'),
            l1_ratio=params.get('l1_ratio', 0.15),
            max_iter=1000, random_state=RANDOM_STATE,
        )
        return AdaBoostClassifier(
            estimator=base,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            random_state=RANDOM_STATE,
        )
    elif name == 'AdaBoost Tree':
        base = DecisionTreeClassifier(
            max_depth=params.get('max_depth', 1),
            random_state=RANDOM_STATE,
        )
        return AdaBoostClassifier(
            estimator=base,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            random_state=RANDOM_STATE,
        )
    elif name == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=params['n_estimators'],
            max_depth=params.get('max_depth', None),
            min_samples_leaf=params.get('min_samples_leaf', 1),
            max_features=params.get('max_features', 'sqrt'),
            class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1,
        )
    elif name == 'LightGBM':
        return LGBMClassifier(
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            max_depth=params.get('max_depth', -1),
            num_leaves=params.get('num_leaves', 31),
            reg_alpha=params.get('reg_alpha', 0.0),
            reg_lambda=params.get('reg_lambda', 0.0),
            min_child_samples=params.get('min_child_samples', 20),
            random_state=RANDOM_STATE, verbose=-1, n_jobs=-1,
        )
    elif name == 'XGBoost':
        return XGBClassifier(
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            max_depth=params.get('max_depth', 3),
            min_child_weight=params.get('min_child_weight', 1),
            gamma=params.get('gamma', 0.0),
            subsample=params.get('subsample', 0.8),
            colsample_bytree=params.get('colsample_bytree', 0.8),
            reg_alpha=params.get('reg_alpha', 0.0),
            reg_lambda=params.get('reg_lambda', 1.0),
            random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0,
        )
    elif name == 'CatBoost':
        return CatBoostClassifier(
            iterations=params['n_estimators'],
            learning_rate=params['learning_rate'],
            depth=params.get('depth', 4),
            l2_leaf_reg=params.get('l2_leaf_reg', 3.0),
            random_strength=params.get('random_strength', 1.0),
            border_count=params.get('border_count', 128),
            random_seed=RANDOM_STATE, verbose=0,
        )
    raise ValueError(f'Unknown model: {name}')

In [ ]:
def make_optuna_objective(name, X, y, lam, skf):
    def objective(trial):
        if name == 'AdaBoost Linear':
            penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
            params = {
                'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 2.0, log=True),
                'alpha':         trial.suggest_float('alpha', 1e-5, 1.0, log=True),
                'penalty':       penalty,
                'l1_ratio':      trial.suggest_float('l1_ratio', 0.1, 0.9) if penalty == 'elasticnet' else 0.15,
            }
        elif name == 'AdaBoost Tree':
            params = {
                'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 2.0, log=True),
                'max_depth':     trial.suggest_int('max_depth', 1, 4),
            }
        elif name == 'Random Forest':
            params = {
                'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
                'max_depth':        trial.suggest_int('max_depth', 2, 15),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
                'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5, 0.7]),
            }
        elif name == 'LightGBM':
            params = {
                'n_estimators':      trial.suggest_int('n_estimators', 50, 500),
                'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                'max_depth':         trial.suggest_int('max_depth', 3, 12),
                'num_leaves':        trial.suggest_int('num_leaves', 8, 128),
                'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
                'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            }
        elif name == 'XGBoost':
            params = {
                'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
                'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                'max_depth':        trial.suggest_int('max_depth', 2, 8),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
                'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
                'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
                'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
                'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            }
        elif name == 'CatBoost':
            params = {
                'n_estimators':    trial.suggest_int('n_estimators', 50, 500),
                'learning_rate':   trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                'depth':           trial.suggest_int('depth', 2, 8),
                'l2_leaf_reg':     trial.suggest_float('l2_leaf_reg', 0.5, 10.0, log=True),
                'random_strength': trial.suggest_float('random_strength', 0.1, 5.0, log=True),
                'border_count':    trial.suggest_int('border_count', 32, 255),
            }
        fold_val_auc, fold_train_auc = [], []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = build_model(name, params)
            try:
                model.fit(X_tr, y_tr)
                fold_val_auc.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
                fold_train_auc.append(roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1]))
            except Exception:
                fold_val_auc.append(np.nan)
                fold_train_auc.append(np.nan)
        val_auc = np.nanmean(fold_val_auc)
        gap     = np.nanmean(fold_train_auc) - val_auc
        return val_auc - lam * gap
    return objective


def cv_evaluate(name, X, y, params, skf):
    fold_val_auc, fold_train_auc, fold_logloss, fold_brier = [], [], [], []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_tr, y_tr)
            proba    = model.predict_proba(X_val)[:, 1]
            proba_tr = model.predict_proba(X_tr)[:, 1]
            fold_val_auc.append(roc_auc_score(y_val, proba))
            fold_train_auc.append(roc_auc_score(y_tr, proba_tr))
            fold_logloss.append(log_loss(y_val, proba))
            fold_brier.append(brier_score_loss(y_val, proba))
        except Exception:
            fold_val_auc.append(np.nan); fold_train_auc.append(np.nan)
            fold_logloss.append(np.nan); fold_brier.append(np.nan)
    valid_val = [v for v in fold_val_auc if not np.isnan(v)]
    return {
        'CV AUC':      np.nanmean(fold_val_auc),
        'CV AUC Std':  np.std(valid_val) if valid_val else np.nan,
        'Train AUC':   np.nanmean(fold_train_auc),
        'Overfit Gap': np.nanmean(fold_train_auc) - np.nanmean(fold_val_auc),
        'Logloss':     np.nanmean(fold_logloss),
        'BrierScore':  np.nanmean(fold_brier),
    }

## Pipeline

**Preprocessing:** Imputer fit on the training set only — no leakage of test-set statistics. StandardScaler applied only for AdaBoost Linear (SGDClassifier is sensitive to feature magnitude; tree-based models are scale-invariant). Both preprocessors stored in `PIPE` for consistent reapplication to any downstream feature subset.

In [ ]:
PIPE = {name: {} for name in MODEL_NAMES}

for name in MODEL_NAMES:
    imputer  = SimpleImputer(strategy='median')
    X_tr_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns, index=X_train.index,
    )
    X_te_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns, index=X_test.index,
    )
    if MODEL_NEEDS_SCALER[name]:
        scaler   = StandardScaler()
        X_tr_imp = pd.DataFrame(
            scaler.fit_transform(X_tr_imp),
            columns=X_tr_imp.columns, index=X_tr_imp.index,
        )
        X_te_imp = pd.DataFrame(
            scaler.transform(X_te_imp),
            columns=X_te_imp.columns, index=X_te_imp.index,
        )
        PIPE[name]['scaler'] = scaler
    PIPE[name]['imputer']       = imputer
    PIPE[name]['X_train_clean'] = X_tr_imp
    PIPE[name]['X_test_clean']  = X_te_imp

print('Preprocessing complete.')
for name in MODEL_NAMES:
    print(f'  {name:20s}  train {PIPE[name]["X_train_clean"].shape}'
          f'  NaN={PIPE[name]["X_train_clean"].isna().sum().sum()}'
          f'  scaled={MODEL_NEEDS_SCALER[name]}')

### Step 1 — Full-feature Optuna tuning

Tune each model on all 26 features before any feature selection. Tuning first ensures that permutation importance in Step 2 reflects a well-regularized model rather than arbitrary default hyperparameters. Objective: **AUC − λ × overfit gap** (λ = 0.3).

In [ ]:
for name in MODEL_NAMES:
    print(f'\n{"="*60}')
    print(f'  Optuna (full features): {name}  [{N_TRIALS_FULL} trials]')
    print(f'{"="*60}')
    X_tr  = PIPE[name]['X_train_clean']
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(
        make_optuna_objective(name, X_tr, y_train, LAM, skf),
        n_trials=N_TRIALS_FULL, show_progress_bar=True,
    )
    res = cv_evaluate(name, X_tr, y_train, study.best_params, skf)
    PIPE[name]['study_full']  = study
    PIPE[name]['params_full'] = study.best_params
    PIPE[name]['cv_full']     = res
    print(f'  CV AUC:      {res["CV AUC"]:.4f} +/- {res["CV AUC Std"]:.4f}')
    print(f'  Train AUC:   {res["Train AUC"]:.4f}')
    print(f'  Overfit Gap: {res["Overfit Gap"]:.4f}')
    print(f'  Best params: {study.best_params}')

print('\n\n-- Full-Feature Optuna Summary --')
rows = []
for name in MODEL_NAMES:
    r = PIPE[name]['cv_full']
    rows.append({
        'Model':       name,
        'CV AUC':      f'{r["CV AUC"]:.4f} +/- {r["CV AUC Std"]:.4f}',
        'Train AUC':   round(r['Train AUC'], 4),
        'Overfit Gap': round(r['Overfit Gap'], 4),
        'Logloss':     round(r['Logloss'], 4),
        'BrierScore':  round(r['BrierScore'], 4),
    })
print(pd.DataFrame(rows).set_index('Model').to_string())

### Step 2 — Cross-validated permutation importance

Importance computed on held-out validation folds — not training data — to avoid inflated scores from memorization. Tree-based models use CV SHAP on the validation fold (exact, captures interactions); AdaBoost Linear uses CV permutation importance. AdaBoost Tree stumps fall back to permutation importance automatically.

In [ ]:
TREE_MODEL_NAMES_2 = {'AdaBoost Tree', 'Random Forest', 'LightGBM', 'XGBoost', 'CatBoost'}

for name in MODEL_NAMES:
    print(f'\n{name}')
    X_tr   = PIPE[name]['X_train_clean']
    params = PIPE[name]['params_full']
    fold_importances = []

    for train_idx, val_idx in skf.split(X_tr, y_train):
        X_f, X_v = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_f, y_f)
            if name in TREE_MODEL_NAMES_2:
                try:
                    # CV SHAP: model trained on fold, SHAP computed on held-out validation
                    explainer = shap.TreeExplainer(model)
                    sv = explainer.shap_values(X_v)
                    if isinstance(sv, list):
                        sv = sv[1]
                    elif sv.ndim == 3:
                        sv = sv[:, :, 1]
                    fold_importances.append(np.abs(sv).mean(axis=0))
                except Exception:
                    # SHAP not supported (e.g. AdaBoost Tree stumps) — fall back to perm importance
                    perm = permutation_importance(
                        model, X_v, y_v,
                        n_repeats=5, random_state=RANDOM_STATE, scoring='roc_auc',
                    )
                    fold_importances.append(perm.importances_mean)
            else:
                # CV permutation importance for AdaBoost Linear
                perm = permutation_importance(
                    model, X_v, y_v,
                    n_repeats=5, random_state=RANDOM_STATE, scoring='roc_auc',
                )
                fold_importances.append(perm.importances_mean)
        except Exception as e:
            print(f'  fold error ({e})')

    mean_imp = np.mean(fold_importances, axis=0)
    std_imp  = np.std(fold_importances, axis=0)
    perm_df  = pd.DataFrame({
        'Feature':    X_tr.columns,
        'Importance': mean_imp,
        'Std':        std_imp,
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    PIPE[name]['perm_full'] = perm_df
    print(perm_df.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(20, 16))
for ax, name in zip(axes.flat, MODEL_NAMES):
    df_p   = PIPE[name]['perm_full'].sort_values('Importance', ascending=True)
    colors = ['#d62728' if v > 0 else '#aec7e8' for v in df_p['Importance']]
    ax.barh(df_p['Feature'], df_p['Importance'], xerr=df_p['Std'],
            color=colors, error_kw={'ecolor': 'gray', 'capsize': 2})
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Mean |SHAP| (tree models) / CV AUC Drop (linear)')
    ax.tick_params(axis='y', labelsize=7)
plt.suptitle('Cross-Validated Feature Importance -- Full Feature Set\n'
             '(CV SHAP for tree models, CV Permutation Importance for AdaBoost Linear)',
             fontsize=14)
plt.tight_layout(); plt.show()

### Step 3 — Dynamic genre consolidation

Tree models (CV SHAP): keep genres **above mean** genre importance — prevents complex models from retaining near-zero SHAP genres. Perm models (AdaBoost): keep importance > 0. Low-signal genres are merged into `artist_genre_other`. Applied to train and test using training-set statistics only.

In [ ]:
SHAP_MODELS = {'Random Forest', 'LightGBM', 'XGBoost', 'CatBoost'}

def consolidate_genres(X_tr, X_te, high_signal, all_genre_cols):
    """Drop low-signal genre dummies and fold them into artist_genre_other."""
    low_signal = [c for c in all_genre_cols if c not in high_signal]
    X_tr_c = X_tr.drop(columns=low_signal).copy()
    X_te_c = X_te.drop(columns=low_signal).copy()
    if low_signal:
        X_tr_c['artist_genre_other'] = (X_tr[low_signal].sum(axis=1) > 0).astype(int)
        X_te_c['artist_genre_other'] = (X_te[low_signal].sum(axis=1) > 0).astype(int)
    return X_tr_c, X_te_c, low_signal

for name in MODEL_NAMES:
    X_tr    = PIPE[name]['X_train_clean']
    X_te    = PIPE[name]['X_test_clean']
    perm_df = PIPE[name]['perm_full']

    all_genre_cols = [c for c in X_tr.columns if c.startswith('artist_genre_')]
    genre_imp      = perm_df.set_index('Feature')['Importance']

    if name in SHAP_MODELS:
        # Keep genres above mean — prevents complex models from retaining near-zero SHAP genres
        genre_vals  = genre_imp[[c for c in all_genre_cols if c in genre_imp.index]]
        genre_mean  = genre_vals.mean()
        high_signal = [c for c in all_genre_cols if genre_imp.get(c, 0) > genre_mean]
    else:
        # Perm importance: > 0 is already conservative enough
        high_signal = [c for c in all_genre_cols if genre_imp.get(c, 0) > 0]

    X_tr_cons, X_te_cons, low_signal = consolidate_genres(X_tr, X_te, high_signal, all_genre_cols)

    PIPE[name]['X_train_cons']       = X_tr_cons
    PIPE[name]['X_test_cons']        = X_te_cons
    PIPE[name]['high_signal_genres'] = high_signal
    PIPE[name]['low_signal_genres']  = low_signal

    print(f'{name}:')
    print(f'  kept {len(high_signal)} genres: {[c.replace("artist_genre_", "") for c in high_signal]}')
    print(f'  merged {len(low_signal)} -> artist_genre_other  |  total features: {X_tr_cons.shape[1]}')

### Step 4 — Forward selection

Greedy addition ordered by Step 2 CV importance. Centrality features compete in the same pool as genre and behavioral features. Tracks CV AUC and overfit gap at each n. Passes two candidates to Step 5: `n_peak` (max CV AUC) and `n_gap` (min overfit gap).

In [ ]:
for name in MODEL_NAMES:
    print(f'\n{"─"*60}\nForward selection: {name}')
    X_tr   = PIPE[name]['X_train_cons']
    X_te   = PIPE[name]['X_test_cons']
    params = PIPE[name]['params_full']

    # Feature ordering from Step 2 CV importance, filtered to consolidated pool
    perm_full_order = PIPE[name]['perm_full'].set_index('Feature')['Importance']
    feature_order   = [
        f for f in perm_full_order.sort_values(ascending=False).index
        if f in X_tr.columns
    ]
    # artist_genre_other is created in Step 3 — append at end so forward selection can evaluate it
    other_col = 'artist_genre_other'
    if other_col in X_tr.columns and other_col not in feature_order:
        feature_order.append(other_col)

    PIPE[name]['feature_order'] = feature_order

    sel_results = []
    start_n = min(3, len(feature_order))
    for n_feats in range(start_n, len(feature_order) + 1):
        feats = feature_order[:n_feats]
        X_sub = X_tr[feats]
        fold_val, fold_tr, fold_ll, fold_bs = [], [], [], []
        for train_idx, val_idx in skf.split(X_sub, y_train):
            X_f, X_v = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
            y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
            model = build_model(name, params)
            try:
                model.fit(X_f, y_f)
                p_v = model.predict_proba(X_v)[:, 1]
                p_t = model.predict_proba(X_f)[:, 1]
                fold_val.append(roc_auc_score(y_v, p_v))
                fold_tr.append(roc_auc_score(y_f, p_t))
                fold_ll.append(log_loss(y_v, p_v))
                fold_bs.append(brier_score_loss(y_v, p_v))
            except Exception:
                fold_val.append(np.nan); fold_tr.append(np.nan)
                fold_ll.append(np.nan);  fold_bs.append(np.nan)
        val_auc = np.nanmean(fold_val); tr_auc = np.nanmean(fold_tr)
        sel_results.append({
            'n_features':  n_feats,
            'CV AUC':      val_auc,
            'Train AUC':   tr_auc,
            'Overfit Gap': tr_auc - val_auc,
            'Logloss':     np.nanmean(fold_ll),
            'BrierScore':  np.nanmean(fold_bs),
        })
        print(f'  n={n_feats:2d} +[{feature_order[n_feats-1][:40]:40s}]  '
              f'AUC={val_auc:.4f}  Gap={tr_auc - val_auc:.4f}')

    df_sel = pd.DataFrame(sel_results).set_index('n_features')
    PIPE[name]['forward_sel'] = df_sel

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, name in zip(axes.flat, MODEL_NAMES):
    df_s   = PIPE[name]['forward_sel']
    n_peak = df_s['CV AUC'].idxmax()
    n_gap  = df_s['Overfit Gap'].idxmin()
    ax.plot(df_s.index, df_s['CV AUC'],      'o-', color='steelblue', lw=2, label='CV AUC')
    ax.plot(df_s.index, df_s['Overfit Gap'], 's--', color='#d62728',  lw=1.5, label='Overfit Gap')
    ax.axvline(n_peak, color='steelblue', linestyle=':', lw=1.5, label=f'n_peak={n_peak}')
    ax.axvline(n_gap,  color='#d62728',   linestyle=':', lw=1.5, label=f'n_gap={n_gap}')
    ax.set_title(name, fontsize=11); ax.set_xlabel('Number of Features')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
plt.suptitle('Forward Selection — CV AUC and Overfit Gap\n(n_peak=blue dashed, n_gap=red dashed)', fontsize=14)
plt.tight_layout(); plt.show()

### Step 5 — Optuna re-tune on candidates + winner selection

Hyperparameters tuned on 26 features may not be optimal for a trimmed subset. Optuna re-tunes on both `n_peak` and `n_gap`. Winner = highest penalized score (CV AUC − λ × gap). **MIN_N = 5** floor prevents pathologically sparse selections.

In [ ]:
MIN_N = 5

for name in MODEL_NAMES:
    print(f'\n{"="*60}\nRe-tune: {name}')
    df_sel    = PIPE[name]['forward_sel']
    feat_ord  = PIPE[name]['feature_order']
    X_tr_cons = PIPE[name]['X_train_cons']
    X_te_cons = PIPE[name]['X_test_cons']

    n_peak = int(df_sel['CV AUC'].idxmax())
    n_gap  = int(df_sel['Overfit Gap'].idxmin())
    candidate_ns = sorted(set([n_peak, n_gap]))
    print(f'  n_peak={n_peak}, n_gap={n_gap}, candidates: {candidate_ns}')

    best_params_by_n = {}
    cv_results_by_n  = {}
    X_train_by_n     = {n: X_tr_cons[feat_ord[:n]] for n in candidate_ns}
    X_test_by_n      = {n: X_te_cons[feat_ord[:n]] for n in candidate_ns}

    for n in candidate_ns:
        print(f'\n  -- n={n}: {feat_ord[:n]}')
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
        study.optimize(
            make_optuna_objective(name, X_train_by_n[n], y_train, LAM, skf),
            n_trials=N_TRIALS_RETUNE, show_progress_bar=True,
        )
        best_params_by_n[n] = study.best_params
        cv_results_by_n[n]  = cv_evaluate(name, X_train_by_n[n], y_train, study.best_params, skf)
        res = cv_results_by_n[n]
        print(f'  CV AUC={res["CV AUC"]:.4f} +/- {res["CV AUC Std"]:.4f}  '
              f'Gap={res["Overfit Gap"]:.4f}  Penalized={res["CV AUC"] - LAM * res["Overfit Gap"]:.4f}')

    # Winner: highest penalized score (CV AUC - LAM * gap) after re-tuning.
    winner_n = max(candidate_ns,
                   key=lambda n: cv_results_by_n[n]['CV AUC'] - LAM * cv_results_by_n[n]['Overfit Gap'])

    # MIN_N guard: override if the penalized winner is below the minimum feature count.
    min_n_overrode = False
    if winner_n < MIN_N:
        eligible = [n for n in candidate_ns if n >= MIN_N]
        if eligible:
            override_n = min(eligible)
            print(f'  MIN_N override: winner n={winner_n} < {MIN_N} → override to n={override_n}')
            winner_n = override_n
            min_n_overrode = True

    PIPE[name]['best_params_by_n'] = best_params_by_n
    PIPE[name]['cv_results_by_n']  = cv_results_by_n
    PIPE[name]['X_train_by_n']     = X_train_by_n
    PIPE[name]['X_test_by_n']      = X_test_by_n
    PIPE[name]['n_optimal']        = winner_n
    PIPE[name]['params_final']     = best_params_by_n[winner_n]
    PIPE[name]['X_train_final']    = X_train_by_n[winner_n]
    PIPE[name]['X_test_final']     = X_test_by_n[winner_n]
    PIPE[name]['cv_result_final']  = cv_results_by_n[winner_n]

    if n_peak == n_gap:
        decision = 'n_peak=n_gap'
    elif min_n_overrode:
        decision = 'MIN_N override'
    elif winner_n == n_gap:
        decision = 'n_gap wins'
    else:
        decision = 'n_peak wins'
    PIPE[name]['step5_decision'] = decision
    print(f'\n  → Winner: n={winner_n}  ({decision})'
          f'  CV AUC={cv_results_by_n[winner_n]["CV AUC"]:.4f}'
          f'  Gap={cv_results_by_n[winner_n]["Overfit Gap"]:.4f}')

# Summary table
print(f'\n{"─"*80}')
print(f'{"Model":<22}  {"n_peak":>6}  {"n_gap":>5}  {"Winner":>6}  '
      f'{"CV AUC":>8}  {"Gap":>6}  Decision')
print('-' * 80)
for name in MODEL_NAMES:
    df_sel = PIPE[name]['forward_sel']
    n_peak = int(df_sel['CV AUC'].idxmax())
    n_gap  = int(df_sel['Overfit Gap'].idxmin())
    n_win  = PIPE[name]['n_optimal']
    res    = PIPE[name]['cv_result_final']
    print(f'{name:<22}  {n_peak:>6}  {n_gap:>5}  {n_win:>6}  '
          f'{res["CV AUC"]:.4f}  {res["Overfit Gap"]:.4f}  {PIPE[name]["step5_decision"]}')

### Step 6 — Centrality ablation (post-hoc)

Tests all 2³ subsets of the centrality features present in the winning set. Selection criterion: raw CV AUC (no penalty) — the overfit penalty was already applied in Step 5. If any subset improves AUC, the feature set and metrics are updated.

In [ ]:
from itertools import combinations as itercombs

CENTRALITY_COLS = [
    'betweenness_centrality_top20_rolling5',
    'harmonic_closeness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
]

short = lambda cols: [c.replace('_centrality_top20_rolling5', '') for c in cols]

for name in MODEL_NAMES:
    X_tr   = PIPE[name]['X_train_final']
    X_te   = PIPE[name]['X_test_final']
    params = PIPE[name]['params_final']
    cent   = [c for c in CENTRALITY_COLS if c in X_tr.columns]

    print(f'\n{"─"*60}\n{name}  (centrality in winner set: {short(cent)})')

    if not cent:
        PIPE[name]['centrality_kept'] = []
        PIPE[name]['centrality_drop'] = []
        print('  no centrality features in winner set — skip')
        continue

    rows = []
    for n_drop in range(len(cent) + 1):
        for dropped in itercombs(cent, n_drop):
            dropped = list(dropped)
            feats   = [c for c in X_tr.columns if c not in dropped]
            res     = cv_evaluate(name, X_tr[feats], y_train, params, skf)
            rows.append({
                '_dropped_cols': dropped,
                'Dropped':       ', '.join(short(dropped)) or 'none (baseline)',
                'CV AUC':        round(res['CV AUC'], 4),
                'Gap':           round(res['Overfit Gap'], 4),
            })

    df_abl = pd.DataFrame(rows).sort_values('CV AUC', ascending=False)
    print(df_abl[['Dropped', 'CV AUC', 'Gap']].to_string(index=False))

    best    = df_abl.iloc[0]
    to_drop = best['_dropped_cols']

    if to_drop:
        X_tr_new = X_tr.drop(columns=to_drop)
        X_te_new = X_te.drop(columns=to_drop)
        new_res  = cv_evaluate(name, X_tr_new, y_train, params, skf)
        PIPE[name]['X_train_final']   = X_tr_new
        PIPE[name]['X_test_final']    = X_te_new
        PIPE[name]['n_optimal']       = X_tr_new.shape[1]
        PIPE[name]['cv_result_final'] = new_res
        print(f'  → dropped {short(to_drop)} '
              f'(CV AUC {best["CV AUC"]:.4f} > baseline {df_abl[df_abl["Dropped"]=="none (baseline)"]["CV AUC"].values[0]:.4f})')
    else:
        print(f'  → keep all centrality (baseline is best at CV AUC={best["CV AUC"]:.4f})')

    cent_kept = [c for c in cent if c not in to_drop]
    PIPE[name]['centrality_kept'] = cent_kept
    PIPE[name]['centrality_drop'] = to_drop

### Final selection summary

Feature set, centrality outcome, and CV metrics for each model after Steps 4–6.

In [ ]:
short_cent = lambda cols: [c.replace('_centrality_top20_rolling5', '') for c in cols]

print(f'{"─"*80}')
print(f'Final Feature Selections (after forward selection + Optuna + centrality ablation)')
print(f'{"─"*80}')
print(f'{"Model":<22}  {"n":>3}  {"CV AUC":>8}  {"Gap":>6}  Centrality kept  Features')
print('-' * 80)
for name in MODEL_NAMES:
    n     = PIPE[name]['n_optimal']
    res   = PIPE[name]['cv_result_final']
    feats = list(PIPE[name]['X_train_final'].columns)
    ck    = short_cent(PIPE[name].get('centrality_kept', []))
    print(f'{name:<22}  {n:>3}  {res["CV AUC"]:.4f}  {res["Overfit Gap"]:.4f}  '
          f'{str(ck):<18}  {feats}')

### Step 7 — Final model evaluation

Fit each model on the full training set (n_optimal features, tuned params). Evaluated **once** on the held-out test set — the only time the test set is used for reporting. Produces per-model: ROC curve, confusion matrix, calibration curve, PR curve, and permutation importance on the training set.

In [ ]:
def compute_lift(y_true, y_proba):
    """Cumulative gain and lift at each probability rank threshold."""
    n         = len(y_true)
    total_pos = np.sum(y_true)
    sorted_idx = np.argsort(y_proba)[::-1]
    y_sorted   = np.array(y_true)[sorted_idx]
    cum_pos    = np.cumsum(y_sorted)
    pop_frac   = np.arange(1, n + 1) / n
    gain       = cum_pos / total_pos
    lift       = gain / pop_frac
    return pop_frac, gain, lift

for name in MODEL_NAMES:
    print(f'\n{"="*60}')
    print(f'  FINAL MODEL: {name}  (n={PIPE[name]["n_optimal"]} features)')
    print(f'{"="*60}')
    X_tr   = PIPE[name]['X_train_final']
    X_te   = PIPE[name]['X_test_final']
    params = PIPE[name]['params_final']

    model = build_model(name, params)
    model.fit(X_tr, y_train)
    PIPE[name]['model_final'] = model

    y_proba_te = model.predict_proba(X_te)[:, 1]
    y_proba_tr = model.predict_proba(X_tr)[:, 1]
    y_pred_50  = (y_proba_te >= 0.5).astype(int)

    PIPE[name]['y_proba_te'] = y_proba_te
    PIPE[name]['y_proba_tr'] = y_proba_tr
    PIPE[name]['test_auc']   = roc_auc_score(y_test, y_proba_te)
    PIPE[name]['train_auc']  = roc_auc_score(y_train, y_proba_tr)
    PIPE[name]['gap']        = PIPE[name]['train_auc'] - PIPE[name]['test_auc']
    PIPE[name]['logloss']    = log_loss(y_test, y_proba_te)
    PIPE[name]['brier']      = brier_score_loss(y_test, y_proba_te)

    print(f'  Test AUC: {PIPE[name]["test_auc"]:.4f}  Train AUC: {PIPE[name]["train_auc"]:.4f}  Gap: {PIPE[name]["gap"]:.4f}')

### Partial Dependence Plots

Shows how predicted hitmaker probability changes as one feature varies while all others are held at their mean. Top 3 features by permutation importance per model. Note: AdaBoost Linear x-axis values are on the StandardScaler scale — interpret direction, not magnitude.

In [ ]:
for name in MODEL_NAMES:
    model     = PIPE[name]['model_final']
    X_tr      = PIPE[name]['X_train_final']
    top_feats = PIPE[name]['perm_final']['Feature'].head(3).tolist()
    scale_note = ' (scaled units)' if MODEL_NEEDS_SCALER[name] else ''

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    try:
        PartialDependenceDisplay.from_estimator(
            model, X_tr, features=top_feats,
            ax=axes, kind='average', grid_resolution=50,
            line_kw={'color': 'steelblue', 'lw': 2},
        )
        plt.suptitle(f'{name} -- Partial Dependence Plots (top 3 features{scale_note})', fontsize=13)
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'{name}: PDP failed -- {e}')
        plt.close()

### Step 8 — OOF threshold tuning

Threshold tuned on out-of-fold (OOF) predictions from the training set — not the test set — to avoid data leakage. Fallback: if the F1-maximizing threshold gives precision < 0.60, find the highest-F1 threshold with precision ≥ 0.60.

In [ ]:
for name in MODEL_NAMES:
    X_tr   = PIPE[name]['X_train_final']
    params = PIPE[name]['params_final']

    oof_proba = np.zeros(len(y_train))
    for train_idx, val_idx in skf.split(X_tr, y_train):
        X_f, X_v = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_f, y_f)
            oof_proba[val_idx] = model.predict_proba(X_v)[:, 1]
        except Exception:
            oof_proba[val_idx] = 0.5

    thresholds = np.arange(0.05, 0.95, 0.01)
    precs = np.array([precision_score(y_train, (oof_proba >= t).astype(int), zero_division=0) for t in thresholds])
    recs  = np.array([recall_score(y_train,    (oof_proba >= t).astype(int), zero_division=0) for t in thresholds])
    f1s   = np.array([f1_score(y_train,        (oof_proba >= t).astype(int), zero_division=0) for t in thresholds])

    best_idx = np.argmax(f1s)
    if precs[best_idx] < 0.60:
        valid_mask = precs >= 0.60
        best_idx   = np.argmax(np.where(valid_mask, f1s, 0)) if valid_mask.any() else np.argmin(np.abs(thresholds - 0.5))

    chosen_thresh = round(thresholds[best_idx], 2)
    PIPE[name]['threshold']  = chosen_thresh
    PIPE[name]['oof_proba']  = oof_proba

    y_proba_te = PIPE[name]['y_proba_te']
    y_pred_t   = (y_proba_te >= chosen_thresh).astype(int)
    PIPE[name]['precision_final'] = precision_score

### Lift Curves & Prediction Tables

Lift curve answers: "How many times better than random is the model at identifying hitmakers when screening the top X% of artists?" Most intuitive metric for resource-allocation decisions. Prediction table shows full test-set ranking sorted by predicted probability with error type.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, name in zip(axes.flat, MODEL_NAMES):
    y_p = PIPE[name]['y_proba_te']
    pop_frac, gain, _ = compute_lift(y_test.values, y_p)
    ax.plot(pop_frac * 100, gain * 100, color='steelblue', lw=2, label=name)
    ax.plot([0, 100], [0, 100], 'k--', lw=1, label='Random')
    ax.fill_between(pop_frac * 100, gain * 100, pop_frac * 100, alpha=0.15, color='steelblue')
    ax.set_xlabel('% of Artists Screened (ranked by predicted probability)')
    ax.set_ylabel('% of Hitmakers Captured')
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100); ax.set_ylim(0, 105)
plt.suptitle('Cumulative Gain (Lift) Curves -- Individual Models', fontsize=14)
plt.tight_layout(); plt.show()

for name in MODEL_NAMES:
    thresh = PIPE[name]['threshold']
    y_p    = PIPE[name]['y_proba_te']
    y_pred = (y_p >= thresh).astype(int)
    pred_table = pd.DataFrame({
        'True Label':      y_test.values,
        'Predicted Prob':  y_p.round(3),
        'Predicted Label': y_pred,
        'Correct':         (y_test.values == y_pred).astype(int),
        'Error Type':      np.where(y_test.values == y_pred, 'Correct',
                           np.where((y_test.values == 0) & (y_pred == 1),
                                    'FP (false alarm)', 'FN (missed hitmaker)')),
    }, index=y_test.index).sort_values('Predicted Prob', ascending=False).reset_index()
    PIPE[name]['pred_table'] = pred_table
    acc = pred_table['Correct'].mean()
    fp  = (pred_table['Error Type'] == 'FP (false alarm)').sum()
    fn  = (pred_table['Error Type'] == 'FN (missed hitmaker)').sum()
    print(f'\n-- {name} (threshold={thresh}) -- Accuracy={acc:.3f}  FP={fp}  FN={fn} --')
    print(pred_table[['index','True Label','Predicted Prob','Predicted Label','Error Type']]
          .head(20).to_string(index=False))

## Cross-model comparison

All models on the same held-out test set with OOF-tuned thresholds. Includes: metrics table, CV AUC ± std vs test AUC chart, model complexity table, lift overlay, disagreement analysis, and SHAP feature importance heatmaps.

In [ ]:
rows = []
for name in MODEL_NAMES:
    n = PIPE[name]['n_optimal']
    r = PIPE[name]['cv_result_final']
    rows.append({
        'Model':       name,
        'N Features':  n,
        'Threshold':   PIPE[name]['threshold'],
        'CV AUC':      f'{r["CV AUC"]:.4f} +/- {r["CV AUC Std"]:.4f}',
        'Test AUC':    round(PIPE[name]['test_auc'], 4),
        'Overfit Gap': round(PIPE[name]['gap'], 4),
        'Log Loss':    round(PIPE[name]['logloss'], 4),
        'Brier Score': round(PIPE[name]['brier'], 4),
        'Precision':   round(PIPE[name]['precision_final'], 3),
        'Recall':      round(PIPE[name]['recall_final'], 3),
        'F1':          round(PIPE[name]['f1_final'], 3),
    })
df_compare = pd.DataFrame(rows).set_index('Model')
print('Cross-Model Comparison (OOF-tuned thresholds):')
print(df_compare.to_string())

palette = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860']
c_map   = {name: c for name, c in zip(MODEL_NAMES, palette)}

# --- Figure 1: CV AUC +/- std vs Test AUC ---
cv_aucs   = [PIPE[n]['cv_result_final']['CV AUC']     for n in MODEL_NAMES]
cv_stds   = [PIPE[n]['cv_result_final']['CV AUC Std'] for n in MODEL_NAMES]
test_aucs = [PIPE[n]['test_auc'] for n in MODEL_NAMES]
colors    = [c_map[n] for n in MODEL_NAMES]

fig0, ax0 = plt.subplots(figsize=(12, 5))
x = np.arange(len(MODEL_NAMES))
w = 0.35
ax0.bar(x - w/2, cv_aucs, w, yerr=cv_stds, capsize=5,
        color=colors, alpha=0.85, label='CV AUC +/- std',
        error_kw={'ecolor': 'black', 'capsize': 5, 'elinewidth': 1.5})
ax0.bar(x + w/2, test_aucs, w,
        color=colors, alpha=0.40, label='Test AUC',
        edgecolor='black', linewidth=0.8)
for i, (cv, std, te) in enumerate(zip(cv_aucs, cv_stds, test_aucs)):
    ax0.text(x[i] - w/2, cv + std + 0.005, f'{cv:.3f}',
             ha='center', va='bottom', fontsize=8)
    ax0.text(x[i] + w/2, te + 0.005, f'{te:.3f}',
             ha='center', va='bottom', fontsize=8)
ax0.set_xticks(x)
ax0.set_xticklabels(MODEL_NAMES, rotation=15, ha='right')
ax0.set_ylim(0.60, 0.90)
ax0.set_ylabel('AUC')
ax0.set_title('CV AUC +/- Std  vs  Test AUC — per Model', fontsize=13, fontweight='bold')
ax0.legend(fontsize=9)
ax0.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# --- Figure 2: Other metrics ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cross-Model Comparison -- OOF-Tuned Thresholds', fontsize=14, fontweight='bold')

w       = 0.13
offsets = np.arange(len(MODEL_NAMES)) - (len(MODEL_NAMES) - 1) / 2
metrics_h = ['Test AUC', 'Precision', 'Recall', 'F1']
metrics_l = ['Overfit Gap', 'Log Loss', 'Brier Score']
x_h = np.arange(len(metrics_h))
x_l = np.arange(len(metrics_l))

for i, name in enumerate(MODEL_NAMES):
    vals_h = [df_compare.loc[name, m] for m in metrics_h]
    vals_l = [df_compare.loc[name, m] for m in metrics_l]
    for ax, vals, xs in [(axes[0], vals_h, x_h), (axes[1], vals_l, x_l)]:
        ax.bar(xs + offsets[i] * w, vals, w, label=name, color=c_map[name], alpha=0.85)
        for k, v in enumerate(vals):
            ax.text(xs[k] + offsets[i] * w, v + 0.003, f'{v:.3f}',
                    ha='center', va='bottom', fontsize=6)

axes[0].set_xticks(x_h); axes[0].set_xticklabels(metrics_h)
axes[0].set_ylim(0, 1.05); axes[0].set_title('Higher is Better')
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

axes[1].set_xticks(x_l); axes[1].set_xticklabels(metrics_l)
axes[1].set_ylim(0, 0.85); axes[1].set_title('Lower is Better')
axes[1].legend(fontsize=8); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Model complexity

Parameters and leaf counts for each final model. Total Leaves = actual leaf nodes across all trees — the most direct measure of model capacity (more leaves = more decision boundaries).

In [ ]:
def get_key_params(name, params):
    if name == 'AdaBoost Linear':
        return dict(n_estimators=params.get('n_estimators'),
                    lr=round(params.get('learning_rate', 0), 4),
                    depth='— (linear)',
                    reg=f"α={params.get('alpha', 0):.1e}")
    elif name == 'AdaBoost Tree':
        return dict(n_estimators=params.get('n_estimators'),
                    lr=round(params.get('learning_rate', 0), 4),
                    depth=params.get('max_depth', 2),
                    reg='—')
    elif name == 'Random Forest':
        return dict(n_estimators=params.get('n_estimators'),
                    lr='—',
                    depth=params.get('max_depth'),
                    reg=f"min_leaf={params.get('min_samples_leaf')}")
    elif name == 'LightGBM':
        return dict(n_estimators=params.get('n_estimators'),
                    lr=round(params.get('learning_rate', 0), 4),
                    depth=params.get('max_depth'),
                    reg=f"λ={params.get('reg_lambda', 0):.2f}")
    elif name == 'XGBoost':
        return dict(n_estimators=params.get('n_estimators'),
                    lr=round(params.get('learning_rate', 0), 4),
                    depth=params.get('max_depth'),
                    reg=f"α={params.get('reg_alpha', 0):.2f} λ={params.get('reg_lambda', 0):.2f}")
    elif name == 'CatBoost':
        return dict(n_estimators=params.get('n_estimators'),
                    lr=round(params.get('learning_rate', 0), 4),
                    depth=params.get('depth'),
                    reg=f"l2={params.get('l2_leaf_reg', 0):.2f}")

def count_total_leaves(name, model, params):
    try:
        if name in ('Random Forest', 'AdaBoost Tree'):
            return sum(t.get_n_leaves() for t in model.estimators_)
        elif name == 'XGBoost':
            df_t = model.get_booster().trees_to_dataframe()
            return int((df_t['Feature'] == 'Leaf').sum())
        elif name == 'LightGBM':
            # num_leaves * num_trees (upper bound; actual may be lower for shallow trees)
            return model.booster_.num_trees() * params.get('num_leaves', 31)
        elif name == 'CatBoost':
            return int(sum(model.get_tree_leaf_counts()))
    except Exception as e:
        return f'err({e})'
    return '—'

rows = []
for name in MODEL_NAMES:
    model  = PIPE[name]['model_final']
    params = PIPE[name]['params_final']
    kp     = get_key_params(name, params)
    rows.append({
        'Model':          name,
        'N Features':     PIPE[name]['n_optimal'],
        'n_estimators':   kp['n_estimators'],
        'Depth':          kp['depth'],
        'Learning Rate':  kp['lr'],
        'Regularization': kp['reg'],
        'Total Leaves':   count_total_leaves(name, model, params),
        'CV AUC':         round(PIPE[name]['cv_result_final']['CV AUC'], 4),
        'Test AUC':       round(PIPE[name]['test_auc'], 4),
        'F1':             round(PIPE[name]['f1_final'], 3),
    })

df_complexity = pd.DataFrame(rows).set_index('Model')
print('Model Parameters & Complexity:')
print(df_complexity.to_string())

### Lift comparison

Overlay of all 6 lift curves on a single plot, plus lift at key screening percentiles (10%, 20%, 30%, 50%). A steeper initial curve = the model concentrates hitmakers more efficiently at the top of its ranking.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name in MODEL_NAMES:
    y_p = PIPE[name]['y_proba_te']
    pop_frac, gain, _ = compute_lift(y_test.values, y_p)
    ax.plot(pop_frac * 100, gain * 100, lw=2, label=name, color=c_map[name])
ax.plot([0, 100], [0, 100], 'k--', lw=1.5, label='Random baseline')
ax.set_xlabel('% of Artists Screened (ranked by predicted probability)', fontsize=12)
ax.set_ylabel('% of Hitmakers Captured', fontsize=12)
ax.set_title('Cumulative Gain (Lift) -- All Models', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100); ax.set_ylim(0, 105)
plt.tight_layout(); plt.show()

print(f'\n{"Model":<22}  {"@10%":>8}  {"@20%":>8}  {"@30%":>8}  {"@50%":>8}')
print('-' * 60)
for name in MODEL_NAMES:
    y_p = PIPE[name]['y_proba_te']
    pop_frac, gain, lift = compute_lift(y_test.values, y_p)
    lifts = []
    for pct in [0.10, 0.20, 0.30, 0.50]:
        idx = min(np.searchsorted(pop_frac, pct), len(lift) - 1)
        lifts.append(lift[idx])
    print(f'{name:<22}  {lifts[0]:>7.2f}x  {lifts[1]:>7.2f}x  {lifts[2]:>7.2f}x  {lifts[3]:>7.2f}x')

### Disagreement analysis

Artists where models disagree most (high std of predicted probabilities across all 6 models). High-disagreement cases are the most informative for understanding where model assumptions diverge.

In [ ]:
short_names = [n[:12] for n in MODEL_NAMES]
all_probas  = {short: PIPE[name]['y_proba_te']
               for short, name in zip(short_names, MODEL_NAMES)}

df_pred_compare = pd.DataFrame(all_probas, index=y_test.index).round(3)
df_pred_compare['True Label']   = y_test.values
df_pred_compare['True Class']   = y_test.map({0.0: 'One-hit Wonder', 1.0: 'Hitmaker'}).values
df_pred_compare['Disagreement'] = df_pred_compare[short_names].std(axis=1).round(3)
df_pred_compare = df_pred_compare.sort_values('Disagreement', ascending=False)

print('Top 20 most disagreed-upon test artists:')
print(df_pred_compare.head(20).to_string())
print('\nBottom 10 most agreed-upon:')
print(df_pred_compare.tail(10).to_string())

### Feature importance heatmap

Cross-model unsigned importance. Tree models: SHAP mean |value| (captures interactions, computed on training set). AdaBoost Linear: CV permutation importance. Normalized to max=1 per model column. Grey = feature not selected by that model.

In [ ]:
TREE_MODEL_NAMES_H = {'AdaBoost Tree', 'Random Forest', 'LightGBM', 'XGBoost', 'CatBoost'}

# Compute SHAP on final feature set for tree models
for name in MODEL_NAMES:
    if name in TREE_MODEL_NAMES_H:
        try:
            model = PIPE[name]['model_final']
            X_tr  = PIPE[name]['X_train_final']
            explainer = shap.TreeExplainer(model)
            sv = explainer.shap_values(X_tr)
            if isinstance(sv, list):
                sv = sv[1]
            elif sv.ndim == 3:
                sv = sv[:, :, 1]
            PIPE[name]['shap_final_abs']    = pd.Series(np.abs(sv).mean(axis=0), index=X_tr.columns)
            PIPE[name]['shap_final_signed'] = pd.Series(sv.mean(axis=0),          index=X_tr.columns)
        except Exception as e:
            print(f'{name}: SHAP failed ({e}), falling back to perm importance')
            PIPE[name]['shap_final_abs']    = None
            PIPE[name]['shap_final_signed'] = None
    else:
        PIPE[name]['shap_final_abs']    = None
        PIPE[name]['shap_final_signed'] = None

all_feats = sorted(set(
    feat for name in MODEL_NAMES
    for feat in PIPE[name]['X_train_final'].columns
))

df_imp = pd.DataFrame(index=all_feats, columns=MODEL_NAMES, dtype=float)
for name in MODEL_NAMES:
    if name in TREE_MODEL_NAMES_H and PIPE[name]['shap_final_abs'] is not None:
        imp = PIPE[name]['shap_final_abs']
    else:
        imp = PIPE[name]['perm_final'].set_index('Feature')['Importance']
    for feat in all_feats:
        df_imp.loc[feat, name] = imp.get(feat, np.nan)

# Normalize: max=1 per model
df_clipped = df_imp.clip(lower=0)
df_norm    = df_clipped.div(df_clipped.max(axis=0), axis=1)
df_norm['_mean'] = df_norm.mean(axis=1)
df_norm = df_norm.sort_values('_mean', ascending=False).drop(columns='_mean')

fig, ax = plt.subplots(figsize=(13, max(6, len(all_feats) * 0.45)))
mask = df_norm.isna()
sns.heatmap(df_norm.astype(float), annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, mask=mask,
            cbar_kws={'label': 'Normalized importance (max=1 per model)'},
            vmin=0, vmax=1, annot_kws={'size': 8})
for (i, j) in zip(*np.where(df_norm.isna().values)):
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='lightgrey', zorder=3))
    ax.text(j + 0.5, i + 0.5, 'n/s', ha='center', va='center', fontsize=8, color='gray')
ax.set_title(
    'Cross-Model Feature Importance (SHAP for tree models, Perm for linear, max=1 per model)\nGrey = not selected',
    fontsize=12,
)
ax.set_xlabel('Model'); ax.set_ylabel('Feature')
plt.tight_layout(); plt.show()

### Signed feature importance

Same heatmap with direction: positive (red) = pushes toward Hitmaker, negative (blue) = pushes toward One-hit Wonder. Features that flip sign across models signal non-linear or context-dependent relationships.

In [ ]:
df_signed = pd.DataFrame(index=all_feats, columns=MODEL_NAMES, dtype=float)
for name in MODEL_NAMES:
    X_tr = PIPE[name]['X_train_final']
    if name in TREE_MODEL_NAMES_H and PIPE[name]['shap_final_signed'] is not None:
        # SHAP mean value already encodes direction
        signed_imp = PIPE[name]['shap_final_signed']
        for feat in all_feats:
            if feat in signed_imp.index:
                df_signed.loc[feat, name] = signed_imp[feat]
    else:
        # AdaBoost Linear: perm importance * sign(corr with target)
        imp = PIPE[name]['perm_final'].set_index('Feature')['Importance']
        for feat in all_feats:
            if feat in X_tr.columns and feat in imp.index:
                corr = np.corrcoef(X_tr[feat].values, y_train.values)[0, 1]
                df_signed.loc[feat, name] = imp[feat] * np.sign(corr)

# Normalize: max absolute value = 1 per model
abs_max        = df_signed.abs().max(axis=0)
df_signed_norm = df_signed.div(abs_max, axis=1)
df_signed_norm = df_signed_norm.reindex(df_norm.index)  # same row order as unsigned heatmap

fig, ax = plt.subplots(figsize=(13, max(6, len(all_feats) * 0.45)))
mask_s = df_signed_norm.isna()
sns.heatmap(df_signed_norm.astype(float), annot=True, fmt='.2f', cmap='RdBu_r',
            linewidths=0.5, ax=ax, mask=mask_s,
            cbar_kws={'label': 'Signed importance (red=->Hitmaker, blue=->One-hit Wonder)'},
            vmin=-1, vmax=1, annot_kws={'size': 8})
for (i, j) in zip(*np.where(df_signed_norm.isna().values)):
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='lightgrey', zorder=3))
    ax.text(j + 0.5, i + 0.5, 'n/s', ha='center', va='center', fontsize=8, color='gray')
ax.set_title(
    'Signed Feature Importance (SHAP direction for tree models, max=1 per model)\n'
    'Red = -> Hitmaker  .  Blue = -> One-hit Wonder  .  Grey = not selected',
    fontsize=12,
)
ax.set_xlabel('Model'); ax.set_ylabel('Feature')
plt.tight_layout(); plt.show()